In [0]:
bronze_path   = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/bronze/'
silver_path   = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/silver/'
gold_path     = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/gold/'
resource_path = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/resource/'

In [0]:

# criar várias tabelas temporárias de forma prática
silver_map = {
   
    "tmp_silver_customers":  f"{silver_path}/customers/",
    "tmp_silver_orders":     f"{silver_path}/orders/",
    "tmp_silver_products":   f"{silver_path}/product/"

}

for view_name, path in silver_map.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )


In [0]:
df_orders_pending = spark.sql("""
with pending as (
select
    customer_id
    ,order_date
    ,sum(quantity) quantity
    ,store_name
    ,status
from tmp_silver_orders
where lower(status) = 'pending'
group by customer_id, store_name, status, order_date
) select 
p.* ,
c.first_name,
c.phone,
c.email
from pending as p
left join tmp_silver_customers as c
on p.customer_id = c.customer_id
where 1=1
and c.email is not null
and c.phone is not null


""" )


#salvar como delta na gold
df_orders_pending.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema', 'true')\
    .save(f'{gold_path}/orders_pending')